# Day 1

## Import files and library

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.dataframe import DataFrame as DF

spark = SparkSession.builder.getOrCreate()

In [ ]:
path = "../data/0_raw/"

path_order_items = path+"order_item.csv"
path_product = path+"product.csv"
path_category = path+"category_translation.csv"

df_order_items = spark.read.csv(path_order_items, header=True, inferSchema=True)
df_product = spark.read.csv(path_product, header=True, inferSchema=True)
df_category = spark.read.csv(path_category, header=True, inferSchema=True)

df_category.show(5)
df_product.show(5)
df_order_items.show(5)

## Affichage des schémas

In [ ]:
df_order_items.printSchema()
df_product.printSchema()
df_category.printSchema()

## Comptage des lignes

In [ ]:
df_order_items.count()

In [ ]:
df_product.count()

In [ ]:
df_category.count()

## Identification Clés de jointures

In [ ]:
common_cols = set(df_order_items.columns) & set(df_product.columns) 
print("Colonnes en commun :", common_cols)

In [ ]:
def analyse_cle(df : DF, col_name: str):
        total      = df.count()
        distincts  = df.select(col_name).distinct().count()
        nulls      = df.filter(df[col_name].isNull()).count()

        print(f"--- Colonne : {col_name} ---")
        print(f"  Total lignes  : {total}")
        print(f"  Valeurs uniques: {distincts}")
        print(f"  Nulls          : {nulls}")
        print(f"  Unicité        : {distincts == total}")
        
analyse_cle(df_order_items, "product_id")
analyse_cle(df_product, "product_id")

## Création de la jointure

In [ ]:
df_joined = df_order_items.join(df_product, on="product_id", how="inner")

print("Lignes df1      :", df_order_items.count())
print("Lignes df2      :", df_product.count())
print("Lignes jointure :", df_joined.count())

# Détecter les doublons introduits par la jointure
df_joined.groupBy("product_id") \
         .agg(F.count("*").alias("n")) \
         .filter(F.col("n") > 1) \
         .show()

## Création des parquets

In [ ]:
#Créer un parquet pour l'ensemble des données
df_joined.write.parquet("../data/1_bronze/order_items_product.parquet", mode="overwrite")